# GWAS Catalog 2025 Processing Script

This notebook contains the scripts used to process the GWAS Catalog SNP-Phenotype Associations 2025 dataset for Harmonizome. It is adapted from Erol Evangelista's [script](https://github.com/MaayanLab/EnrichrPythonScripts/blob/master/Enrichr/gwas_catalog_2025.ipynb) to process the library for Enrichr. The associations file was downloaded from the GWAS Catalog [portal](https://www.ebi.ac.uk/gwas/downloads) on 10/8/25.

In [ ]:
import pandas as pd
import datetime
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import re
import scipy.spatial.distance as dist
import seaborn as sns
import sys
import json
import scanpy as sc
from tqdm import tqdm

# UMAP
from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()

from IPython.display import display, HTML, Markdown
sys.setrecursionlimit(100000)

import pyarrow.feather as feather
from scipy.stats import rankdata, zscore

In [ ]:
sn = "Homo_sapiens"
organism = "Mammalia/%s"%sn
url = 'ftp://ftp.ncbi.nih.gov/gene/DATA/GENE_INFO/{}.gene_info.gz'.format(organism)
file = '{}.gene_info.tsv'.format(organism)

In [ ]:
def fetch_save_read(url, file, reader=pd.read_csv, sep='\t', **kwargs):
  ''' Download file from {url}, save it to {file}, and subsequently read it with {reader} using pandas options on {**kwargs}.
  '''
  if not os.path.exists(file):
    if os.path.dirname(file):
      os.makedirs(os.path.dirname(file), exist_ok=True)
    df = reader(url, sep=sep, index_col=None)
    df.to_csv(file, sep=sep, index=False)
  return pd.read_csv(file, sep=sep, **kwargs)


In [ ]:
ncbi_gene = fetch_save_read(url, file)
ncbi_gene.head()

In [ ]:
def maybe_split(record):
    ''' NCBI Stores Nulls as '-' and lists '|' delimited
    '''
    if record in {'', '-'}:
        return set()
    return set(record.split('|'))

def supplement_dbXref_prefix_omitted(ids):
    ''' NCBI Stores external IDS with Foreign:ID while most datasets just use the ID
    '''
    for id in ids:
        # add original id
        yield id
        # also add id *without* prefix
        if ':' in id:
            yield id.split(':', maxsplit=1)[1]

ncbi_gene['All_synonyms'] = [
    set.union(
      maybe_split(gene_info['Symbol']),
      maybe_split(gene_info['Symbol_from_nomenclature_authority']),
      maybe_split(str(gene_info['GeneID'])),
      maybe_split(gene_info['Synonyms']),
      maybe_split(gene_info['Other_designations']),
      maybe_split(gene_info['LocusTag']),
      set(supplement_dbXref_prefix_omitted(maybe_split(gene_info['dbXrefs']))),
    )
    for _, gene_info in ncbi_gene.iterrows()
  ]

ncbi_gene['All_synonyms'] = [
    set.union(
      maybe_split(gene_info['Symbol']),
      maybe_split(gene_info['Symbol_from_nomenclature_authority']),
      maybe_split(str(gene_info['GeneID'])),
      maybe_split(gene_info['Synonyms']),
      maybe_split(gene_info['Other_designations']),
      maybe_split(gene_info['LocusTag']),
      set(supplement_dbXref_prefix_omitted(maybe_split(gene_info['dbXrefs']))),
    )
    for _, gene_info in ncbi_gene.iterrows()
  ]

synonyms, gene_id = zip(*{
    (synonym, gene_info['Symbol'])
    for _, gene_info in ncbi_gene.iterrows()
    for synonym in gene_info['All_synonyms']
  })
ncbi_lookup_syn = pd.Series(gene_id, index=synonyms)
symbols, cap, gene_id = zip(*{
    (gene_info['Symbol'], gene_info['Symbol'].upper(), gene_info['Symbol'])
    for _, gene_info in ncbi_gene.iterrows()
  })
ncbi_lookup_sym = pd.Series(gene_id, index=symbols)
ncbi_lookup_sym_cap = pd.Series(gene_id, index=cap)

index_values = ncbi_lookup_syn.index.value_counts()
ambiguous = index_values[index_values > 1].index
ncbi_lookup_syn_disambiguated = ncbi_lookup_syn[(
(ncbi_lookup_syn.index == ncbi_lookup_syn) | (~ncbi_lookup_syn.index.isin(ambiguous))
)]
all_genes = {}
ncbi_lookup_sym = ncbi_lookup_sym.to_dict()
ncbi_lookup_sym_cap = ncbi_lookup_sym_cap.to_dict()
ncbi_lookup_syn_disambiguated = ncbi_lookup_syn_disambiguated.to_dict()
def gene_lookup(gene):
    gene_id = ncbi_lookup_sym.get(gene)
    if gene_id: return str(gene_id)
    gene_id = ncbi_lookup_sym_cap.get(gene)
    if gene_id: return str(gene_id)
    return str(ncbi_lookup_syn_disambiguated.get(gene))

In [ ]:
gene_info = ncbi_gene[(ncbi_gene['#tax_id']==9606) & (ncbi_gene['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

In [ ]:
df = pd.read_csv('gwas_catalog_v1.0-associations_e115_r2025-09-29.tsv', sep="\t")
df.head()

In [ ]:
cols = ['DISEASE/TRAIT', 'REPORTED GENE(S)',   'PVALUE_MLOG']
df = df[cols]
print(len(set(df['DISEASE/TRAIT'])))
df.head()

In [ ]:
all_genes = {}
for genes in tqdm(df['REPORTED GENE(S)'].unique()):
    if type(genes) == str:
        for g1 in genes.split(","):
            for g in g1.split("|"):    
                gene = g.strip()
                if gene not in all_genes:
                    all_genes[gene] = ''
len(all_genes)

In [ ]:
for gene in tqdm(all_genes):
    all_genes[gene] = gene_lookup(gene)

with open("out/all_genes.json", 'w') as o:
    o.write(json.dumps(all_genes))

In [ ]:
gwas_df = df.copy()
gwas_df['DISEASE/TRAIT'] = gwas_df['DISEASE/TRAIT'].str.replace('ö','o')
gwas_df['REPORTED GENE(S)'] = gwas_df['REPORTED GENE(S)'].str.split(',')
gwas_df = gwas_df.explode('REPORTED GENE(S)')
gwas_df['REPORTED GENE(S)'] = gwas_df['REPORTED GENE(S)'].str.split('|')
gwas_df = gwas_df.explode('REPORTED GENE(S)')
gwas_df['REPORTED GENE(S)'] = gwas_df['REPORTED GENE(S)'].str.strip().map(all_genes).map(synonyms)
gwas_df = gwas_df[gwas_df['REPORTED GENE(S)']!='None'].dropna(subset='REPORTED GENE(S)')
gwas_df['PVALUE_MLOG'] = gwas_df['PVALUE_MLOG'].div(10).clip(0,10)
gwas_df.columns = ['Phenotype', 'Gene', 'Score']
gwas_df = gwas_df.drop_duplicates(subset=['Phenotype', 'Gene']).dropna()
gwas_df

In [ ]:
gwas_df.nunique()

## Prepare Data for SQL Ingestion

### Dataset

In [ ]:
# id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, positive_association, negative_association, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural, is_archived
(192, 'GWAS Catalog SNP-Phenotype Associations 2025', 'SNP-Phenotype Associations 2025', 'SNP-phenotype association p-values curated from published GWAS', 'gene-phenotype associations by manual GWAS curation', 'genes associated with the {0} phenotype in GWAS datasets from the GWAS Catalog SNP-Phenotype Associations 2025 dataset.', 'sets of genes associated with phenotypes in GWAS datasets from the GWAS Catalog SNP-Phenotype Associations 2025 dataset.', 'phenotypes associated with {0} gene in GWAS datasets from the GWAS Catalog SNP-Phenotype Associations 2025 dataset.',0, 1, '2025-10-09', 'gwascatalog25', 12584, 32, 19, 1, 15, 3, 'genetic association by data aggregation from genome-wide association and other genetic association studies', 'curated experimental data', 'high throughput, data driven', 'phenotypes')

### Publication

In [ ]:
#(id, long_citation, short_citation, url, pmid, pubmed_url, first_author_last_name, first_author_initials, journal_abbreviation, year, title, volume, pages)
(174, 'Cerezo M, et al. (2025) The NHGRI-EBI GWAS Catalog: standards for reusability, sustainability and diversity. Nucleic Acids Res. 53:D998-1005', 'Cerezo, Nucleic Acids Res, 2025', 'dx.doi.org/10.1093/nar/gkae1070', 39530240, 'https://pubmed.ncbi.nlm.nih.gov/39530240', 'Cerezo', 'M', 'Nucleic Acids Res', 2025, 'The NHGRI-EBI GWAS Catalog: standards for reusability, sustainability and diversity', 53, 'D998-1005')

(280, 192, 174)
(281, 192, 115)

### Gene

In [ ]:
ncbi_gene_info = ncbi_gene.set_index('Symbol')
genes = pd.read_csv('../../tables/gene.txt', sep='\t')
genes['symbol'] = genes['symbol'].str.upper()
genes = genes.set_index('symbol')
genefks = genes['id'].to_dict()

index = 58934

for gene in gwas_df['Gene'].unique():
    if gene.upper() not in genes.index:
        geneid, genedesc = ncbi_gene_info.loc[gene, ['GeneID', 'description']]
        print((index, gene.upper(), geneid, genedesc, f'https://ncbi.nlm.nih.gov/gene/{geneid}'), end=',\n')
        genefks[gene.upper()] = index
        index += 1

### Attribute

In [ ]:
attributes = pd.read_csv('../../tables/attribute.txt', sep='\t')
attributes['name_from_naming_authority'] = attributes['name_from_naming_authority'].str.lower().str.strip()
attributes = attributes.set_index('name_from_naming_authority')
attributefks = attributes['id'].to_dict()
gwas_mapping = pd.read_csv('gwas-efo-trait-mappings.tsv', sep='\t', index_col='Disease trait')['EFO URI'].to_dict()
index = 559166

for phenotype in gwas_df['Phenotype'].unique():
    if phenotype.lower() not in attributes.index:
        phen_uri = gwas_mapping.get(phenotype, '')
        print((index, phenotype, phen_uri, 56), end=',\n')
        attributefks[phenotype.lower()] = index
        index += 1

### Gene Set

In [ ]:
index = 139300000
genesetfks = {}

for phenotype in gwas_df['Phenotype'].unique():
    phen_uri = gwas_mapping.get(phenotype, '')
    print((index, phenotype, phen_uri, 192, 15, attributefks[phenotype.lower()]), end=',\n')
    genesetfks[phenotype] = index
    index += 1

### Associations

In [ ]:
index = 97000000

associations = gwas_df.copy()
associations['Gene'] = associations['Gene'].apply(lambda x: genefks[x])
associations['Phenotype'] = associations['Phenotype'].apply(lambda x: genesetfks[x])
associations['Threshold'] = 1
associations = associations.get(['Gene', 'Phenotype', 'Score', 'Threshold'])
associations.columns = ['gene_fk', 'gene_set_fk', 'standardized_value', 'threshold_value']
associations = associations.drop_duplicates().reset_index(drop=True)
associations.index += index
associations = associations.rename_axis('id')
associations.to_csv('../../harmonizome-update/gwascatalog25.csv')
associations

## Create Downloads

In [ ]:
output_path = 'downloads'

### Binary Matrix

In [ ]:
binarymatrix = pd.crosstab(gwas_df['Gene'], gwas_df['Phenotype']).replace(np.nan, 0)
binarymatrixT = binarymatrix.T
binarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
binarymatrix

### Gene Attribute Edge List

In [ ]:
edgelist = gwas_df.copy()
edgelist['Gene ID'] = edgelist['Gene'].map(geneids).astype(int)
edgelist['Threshold Value'] = 1
edgelist = edgelist[['Gene', 'Gene ID', 'Phenotype', 'Score', 'Threshold Value']].reset_index(drop=True)
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['Phenotype']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(binarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=binarymatrix.index, columns=binarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(binarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=binarymatrixT.index, columns=binarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Standardized Matrix

In [ ]:
standardizedmatrix = pd.crosstab(gwas_df['Gene'], gwas_df['Phenotype'], values=gwas_df['Score'], aggfunc=max).replace(np.nan, 0)
standardizedmatrix.to_csv(f'{output_path}/gene_attribute_matrix_standardized.txt.gz', sep='\t', compression='gzip')
standardizedmatrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for pert in attributeslist.index:
    pert = attributeslist.loc[pert]
    nodes[f"GWASCatlog_{pert['Phenotype']}"] = {
        "type":"phenotype",
        "properties": {
            "label":pert['Phenotype'],
            "id":f"GWASCatlog_{pert['Phenotype']}"
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    edges.append({
        "source": int(edge['Gene ID']),
        "relation": "has phenotype",
        "target": f"GWASCatlog_{edge['Phenotype']}",
        "properties":{
            "id": str(edge['Gene ID'])+":"+f"GWASCatlog_{edge['Phenotype']}",
            "source_label":edge['Gene'],
            "source_id": int(edge['Gene ID']),
            "target_id": f"GWASCatlog_{edge['Phenotype']}",
            "target_label":edge['Phenotype'],
            "directed":True,
            "standardized":edge['Score'],
            "threshold":1,
        }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/gwascatalog25.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    
    print('', file=f)
    for edge in edges:
        print(f"gene:{edge['properties']['source_id']}", 'RO:0002200', f"<{edge['properties']['target_id']}>", end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/gwascatalog25.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
namespace = {'gene':'NCBI Entrez', 'phenotype':'GWAS Catalog'}
nodeframe['namespace'] = nodeframe['type'].map(namespace)
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/gwascatalog25_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['standardized'] = edgeframe['properties'].apply(lambda x: x['standardized'])
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe = edgeframe.get(['source', 'relation', 'target','standardized', 'threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/gwascatalog25_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
sns.clustermap(binarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
ax = sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[term] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_crisp.gmt', 'r'))
scatterdir = 'images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the GWAS Catalog SNP-Phenotype Associations 2025 Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 4, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=30,mindist=0.1
     #,spread=0.5
     ,maxdf=0.5
     ,mindf=1
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in GWAS Catalog SNP-Phenotype Associations 2025 Library')
save(plot)